# 13.7 Manipulating character strings

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The ability to work with arbitrary sets of character strings is one of the key advantages
of AMPL scripting. We describe here the string concatenation operator and several
functions for building up string-valued expressions that can be used anywhere that set
members can appear in AMPL statements. Further details are provided in Section A.4.2,
and Table A-4 summarizes all of the string functions.

We also show how string expressions may be used to specify character strings that
serve purposes other than being set members. This feature allows an AMPL script to, for
example, write a different file or set different option values in each pass through a loop,
according to information derived from the contents of the loop indexing sets.

**String functions and operators**

The concatenation operator & takes two strings as operands, and returns a string consisting
of the left operand followed by the right operand. For example, given the sets
NUTR and FOOD defined by diet.mod and diet2.dat (Figures 2-1 and 2-3), you
could use concatenation to define a set NUTR_FOOD whose members represent nutrientfood
pairs:

```ampl
ampl: model diet.mod;
ampl: data diet2.dat;
ampl: display NUTR, FOOD;
set NUTR := A B1 B2 C NA CAL;
set FOOD := BEEF CHK FISH HAM MCH MTL SPG TUR;
ampl: set NUTR_FOOD := setof {i in NUTR,j in FOOD} i & "_" & j;
ampl: display NUTR_FOOD;
set NUTR_FOOD :=
A_BEEF    B1_BEEF   B2_BEEF   C_BEEF    NA_BEEF   CAL_BEEF
A_CHK     B1_CHK    B2_CHK    C_CHK     NA_CHK    CAL_CHK
A_FISH    B1_FISH   B2_FISH   C_FISH    NA_FISH   CAL_FISH
A_HAM     B1_HAM    B2_HAM    C_HAM     NA_HAM    CAL_HAM
A_MCH     B1_MCH    B2_MCH    C_MCH     NA_MCH    CAL_MCH
A_MTL     B1_MTL    B2_MTL    C_MTL     NA_MTL    CAL_MTL
A_SPG     B1_SPG    B2_SPG    C_SPG     NA_SPG    CAL_SPG
A_TUR     B1_TUR    B2_TUR    C_TUR     NA_TUR    CAL_TUR;
```

This is not a set that you would normally want to define, but it might be useful if you
have to read `data` in which strings like "B2_BEEF" appear.
Numbers that appear as arguments to & are automatically converted to strings. As an
example, for a multi-week `model` you can create a set of generically-named periods
WEEK1, WEEK2, and so forth, by declaring:

```ampl
param T integer > 1;
set WEEKS ordered = setof {t in 1..T} "WEEK" & t;
```

Numeric operands to & are always converted to full precision (or equivalently, to %.0g
format) as defined in [Section A.16](#missing) <!--- xTODO missing reference --->. The conversion thus produces the expected results
for concatenation of numerical constants and of indices that run over sets of integers or
constants, as in our examples. Full precision conversion of computed fractional values
may sometimes yield surprising results, however. The following variation on the preceding
example would seem to create a set of members WEEK0.1, WEEK0.2, and so forth:

```ampl
param T integer > 1;
set WEEKS ordered = setof {t in 1..T} "WEEK" & 0.1*t;
```

But the actual set comes out differently:

```ampl
ampl: let T := 4;
ampl: display WEEKS;
set WEEKS :=
WEEK0.1                              WEEK0.30000000000000004
WEEK0.2                              WEEK0.4;
```

Because 0.1 cannot be stored exactly in a `binary` representation, the value of 0.1*3 is
slightly different from 0.3 in "full" precision. There is no easy way to predict this
behavior, but it can be prevented by specifying an explicit conversion using sprintf.
The sprintf function does format conversions in the same way as printf (Section
A.16), except that the resulting formatted string is not sent to an output stream, but
instead becomes the function's return value. For our example, "WEEK" & 0.1*t could
be replaced by sprintf("WEEK%3.1f",0.1*t).

The length string function takes a string as argument and returns the number of
characters in it. The match function takes two string arguments, and returns the first
position where the second appears as a substring in the first, or zero if the second never
appears as a substring in the first. For example:

```ampl
ampl: display {j in FOOD} (length(j), match(j,"H"));
:    length(j) match(j, 'H')    :=
BEEF      4           0
CHK       3           2
FISH      4           4
HAM       3           1
MCH       3           3
MTL       3           0
SPG       3           0
TUR       3           0
;
```

The substr function takes a string and one or two integers as arguments. It returns a
substring of the first argument that begins at the position given by the second argument; it
has the length given by the third argument, or extends to the end of the string if no third
argument is given. An empty string is returned if the second argument is greater than the
length of the first argument, or if the third argument is less than 1.
As an example combining several of these functions, suppose that you want to use the
`model` from diet.mod but to supply the nutrition amount `data` in a `table` like this:

```ampl
param: NUTR_FOOD: amt_nutr :=
              A_BEEF      60
              B1_BEEF     10
              CAL_BEEF   295
              CAL_CHK    770
              ...
```

Then in addition to the declarations for the parameter amt used in the `model`,

```ampl
set NUTR;
set FOOD;
param amt {NUTR,FOOD} >= 0;
```

you would declare a set and a parameter to hold the `data` from the "nonstandard" `table`:

```ampl
set NUTR_FOOD;
param amt_nutr {NUTR_FOOD} >= 0;
```

To use the `model`, you need to write an assignment of some kind to get the `data` from set
NUTR_FOOD and parameter amt_nutr into sets NUTR and FOOD and parameter amt.
One solution is to extract the sets first, and then convert the parameters:

```ampl
set NUTR = setof {ij in NUTR_FOOD}
                            substr(ij,1,match(ij,"_")-1);
set FOOD = setof {ij in NUTR_FOOD}
                            substr(ij,match(ij,"_")+1);
param amt {i in NUTR, j in FOOD} = amt_nutr[i & "_" & j];
```

As an alternative, you can extract the sets and parameters together with a script such as
the following:

```ampl
param iNUTR symbolic;
param jFOOD symbolic;
param upos > 0;
let NUTR := {};
let FOOD := {};
for {ij in NUTR_FOOD} {
       let upos := match(ij,"_");
       let iNUTR := substr(ij,1,upos-1);
       let jFOOD := substr(ij,upos+1);
       let NUTR := NUTR union {iNUTR};
       let FOOD := FOOD union {jFOOD};
       let amt[iNUTR,jFOOD] := amt_nutr[ij];
}
```

Under either alternative, errors such as a missing "_" in a member of NUTR_FOOD are
eventually signaled by error messages.
AMPL provides two other functions, sub and gsub, that look for the second argument
in the first, like match, but that then substitute a third argument for either the first
occurrence (sub) or all occurrences (gsub) found. The second argument of all three of
these functions is actually a regular expression; if it contains certain special characters, it
is interpreted as a pattern that may match many sub-strings. The pattern "ˆB[0-9]+_",
for example, matches any sub-string consisting of a B followed by one or more digits and
then an underscore, and occurring at the beginning of a string. Details of these features
are given in Section A.4.2.

**String expressions in AMPL commands**

String-valued expressions may appear in place of literal strings in several contexts: in
filenames that are part of commands, including `model`, `data`, and commands, and in
filenames following > or >> to specify redirection of output; in values assigned to AMPL
options by an option command; and in the string-list and the database row and column
names specified in a `table` statement. In all such cases, the string expression must be
identified by enclosing it in parentheses.

Here is an example involving filenames. This script uses a string expression to specify
files for a `data` statement and for the redirection of output from a `display` statement
:

```ampl
model diet.mod;
set CASES = 1 .. 3;
for {j in CASES} {
       reset data;
       data ("diet" & j & ".dat");
       solve;
       display Buy >("diet" & j & ".out");
}
```

The result is to `solve` diet.mod with a series of different `data` files diet1.dat,
diet2.dat, and diet3.dat, and to save the solution to files diet1.out,
diet2.out, and diet3.out. The value of the index `j` is converted automatically
from a number to a string as previously explained.

The following script uses a string expression to specify the value of the option
cplex_options, which contains directions for the CPLEX solver:

```ampl
model sched.mod;
data sched.dat;
option solver cplex;
set DIR1 = {"primal","dual"};
set DIR2 = {"primalopt","dualopt"};
for {i in DIR1, j in DIR2} {
       option cplex_options (i & " " & j);
       solve;
}
```

The loop in this script solves the same problem four times, each using a different pairing
of the directives primal and dual with the directives primalopt and dualopt.

Examples of the use of string expressions in the `table` statement, to work with multiple
database files, tables, or columns, are presented in Section 10.6.